# FASE 8 (opcional) — RED NEURONAL: PERCEPTRÓN MULTICAPA (MLP)

**Objetivo:** comprobar si una red neuronal mejora a los modelos ya ajustados en `Modelado .ipynb`
(referencia a batir: `HGB_ajustado`, RMSE log ≈ 0.4246 en los 5 folds de comparación).

**Por qué un MLP y no una RNN, una CNN o un Transformer:** cada anuncio es una fila de una tabla, sin orden
temporal ni estructura espacial en rejilla. Las RNN necesitan secuencias y las CNN imágenes; el MLP (capas
densas totalmente conectadas) es la red que corresponde a datos tabulares. La literatura (Grinsztajn et al.,
2022) indica que en datos tabulares de este tamaño los árboles con *boosting* suelen ganar a las redes, así que
el resultado esperado es que el MLP **no** supere a HGB; el experimento sirve para comprobarlo con el mismo
protocolo y no por suposición.

**Mismo protocolo que el resto de modelos:**
- Train de modelado y los **5 folds de comparación** (repetición 0 de `outputs/folds_cv.csv`), agrupados por
  anfitrión y estratificados por `room_type × district`. Son los mismos folds que los demás modelos, así que las
  comparaciones son pareadas.
- Todo lo que aprende de los datos (escalado, imputación, codificación del barrio, parada temprana) se ajusta
  **dentro del train de cada fold**.
- Cada candidato se guarda en el registro de experimentos con sus hiperparámetros. Si se vuelve a ejecutar,
  se reutiliza lo ya calculado, y si Colab se desconecta basta con relanzar las celdas.
- El test no se toca: es de la Fase 9.

## 0. ENTORNO: GOOGLE COLAB

1. Sube la carpeta **`Entrenamiento Google Colab`** entera a la raíz de tu Google Drive (*Mi unidad*).
2. Abre este notebook desde Drive con Google Colab.
3. *Entorno de ejecución → Cambiar tipo de entorno de ejecución → **GPU T4***.
4. *Entorno de ejecución → Ejecutar todas*. Los resultados se escriben en
   `outputs/experimentos_mlp.csv` de la carpeta de Drive **después de cada candidato**. Si la sesión se corta,
   se vuelve a ejecutar todo y solo se entrena lo que falta.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# pycountry lo importa preparacion_datos.py (func_modelado depende de el); torch y scikit-learn ya vienen en Colab
%pip install -q pycountry

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display

PROJECT_ROOT = Path('/content/drive/MyDrive/Entrenamiento Google Colab')  # cambiar si se sube a otra carpeta
assert PROJECT_ROOT.exists(), f'No encuentro {PROJECT_ROOT}: ajusta PROJECT_ROOT'
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
from func_modelado import *
from func_red_neuronal import RegresorMLP, preprocesador_mlp

DATA_DIR = PROJECT_ROOT / 'data'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
RUTA_REGISTRO = OUTPUT_DIR / 'experimentos_mlp.csv'   # lo que se entrena aqui; se lleva al proyecto al acabar
RUTA_REFERENCIA = OUTPUT_DIR / 'experimentos.csv'     # copia del registro del proyecto, solo se lee

if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('SIN GPU: Entorno de ejecucion > Cambiar tipo de entorno de ejecucion > GPU')
pd.set_option('display.max_columns', 40)

## 1. DATOS, FOLDS Y VARIABLES

Las variables son las seleccionadas en la Fase 4 (§7.3 de `Modelado .ipynb`). Se usa el conjunto de **Ridge**
(71 variables), por la misma regla que en §8: los modelos que trabajan con combinaciones lineales de variables
escaladas (kNN, SVR y ahora el MLP, cuya primera capa es exactamente eso) usan el conjunto de Ridge, y los
árboles el de HGB. Las listas están en `data/variables_seleccionadas.json`, sacadas del registro de
experimentos para no tener que reejecutar la selección.

In [ ]:
import json

train, bloques = cargar_modelado('train')
folds = cargar_folds(train, OUTPUT_DIR / 'folds_cv.csv', 3)
FOLDS_MODELOS = [f for f in folds if f.repeticion == 0]

VARS_MLP = json.loads((DATA_DIR / 'variables_seleccionadas.json').read_text(encoding='utf-8'))['Ridge']
assert set(VARS_MLP) <= set(train.columns)
print(f'{len(train)} anuncios de train | {len(FOLDS_MODELOS)} folds | {len(VARS_MLP)} variables')

## 2. EL MODELO

**Preprocesado** (`preprocesador_mlp`): es el de Ridge y SVR (`preprocesador_lineal`: one-hot, log de conteos
y de la estancia, imputación con indicador y **escalado**), sin splines ni interacciones, porque la red aprende
la no linealidad por sí misma. Escalar es obligatorio: el descenso por gradiente se atasca si las entradas
tienen escalas muy distintas.

**El barrio** (≈220 categorías) se trata de dos formas, y la búsqueda elige:
- *target encoding* (`dim_embedding = 0`), igual que en Ridge y SVR;
- **embedding** (`dim_embedding = 8 o 16`), la opción propia de las redes: cada barrio recibe un vector que se
  aprende junto con el resto de pesos. Los barrios con menos de 10 anuncios en el train del fold comparten
  vector, y los no vistos tienen uno propio.

**Red** (`RegresorMLP`): capas densas → activación → *dropout*, y una neurona de salida lineal. Se entrena con
AdamW y pérdida MSE sobre el `log(price)` estandarizado, que equivale a minimizar el RMSE del registro.

**Parada temprana:** en cada fold se aparta un 10 % del train, al azar. Se para tras 20 épocas sin mejorar la
pérdida en ese 10 % y se recuperan los pesos de la mejor época. Si la pérdida se estanca 5 épocas, la tasa de
aprendizaje se divide entre 2. El número de épocas no es un hiperparámetro, lo decide la parada. El apartado es
por filas y no por anfitrión (el estimador no ve `host_id`): los anuncios gemelos de un mismo anfitrión pueden
retrasar un poco la parada. Es la misma fuga menor que se aceptó para el TargetEncoder (H-040), y no afecta al
RMSE de validación, que sale del fold separado por anfitrión.

### 2.1 Prueba de humo: un fold, una configuración

Antes de lanzar la búsqueda se comprueba que la red aprende y cuánto tarda un ajuste. Esta celda no se registra.

In [ ]:
import matplotlib.pyplot as plt
from time import perf_counter

f0 = FOLDS_MODELOS[0]
X, y = train[VARS_MLP], train[TARGET]
prueba = make_pipeline(preprocesador_mlp(VARS_MLP, 'embedding'), RegresorMLP(capas=(256, 128), dim_embedding=8))
inicio = perf_counter()
prueba.fit(X.iloc[f0.train], y.iloc[f0.train])
t_prueba = perf_counter() - inicio
mlp_prueba = prueba[-1]
print(f'Ajuste: {t_prueba:.0f} s en {mlp_prueba.dispositivo_} | mejor epoca: {mlp_prueba.epocas_[0]}')
print(metricas(y.iloc[f0.validacion], prueba.predict(X.iloc[f0.validacion])))

historial = mlp_prueba.historiales_[0]
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(historial.index + 1, np.sqrt(historial['mse_ajuste']), label='ajuste (90 % del train del fold)')
ax.plot(historial.index + 1, np.sqrt(historial['mse_parada']), label='parada (10 % apartado)')
ax.axvline(mlp_prueba.epocas_[0], color='grey', ls=':', label='mejor epoca')
ax.set(xlabel='epoca', ylabel='RMSE (target estandarizado)', title='Curva de aprendizaje, fold 0')
ax.legend()
plt.show()

## 3. BÚSQUEDA DE HIPERPARÁMETROS

Como en §10, es una **búsqueda aleatoria** (`ParameterSampler`) y no una rejilla completa. Con 7 dimensiones,
la rejilla completa tendría miles de combinaciones; la aleatoria prueba muchos valores distintos de cada
hiperparámetro con el mismo presupuesto (Bergstra y Bengio, 2012). **El presupuesto se fija aquí, antes de ver
ningún resultado: 60 candidatos × 5 folds.** Con 60 muestras, la probabilidad de que al menos una caiga en el 5 %
mejor del espacio es 1 − 0,95⁶⁰ ≈ 95 %.

| Hiperparámetro | Valores | Por qué ese rango |
|---|---|---|
| `capas` | (64), (256), (128, 64), (256, 128), (512, 256), (256, 128, 64), (512, 256, 128) | De 1 a 3 capas ocultas, en forma de embudo. Con ≈20 000 anuncios por fold y ≈90 entradas, más capas o anchos mayores solo añaden capacidad para memorizar |
| `activacion` | ReLU, SiLU | ReLU es la estándar; SiLU es suave y a veces va mejor con regresión |
| `dropout` | 0, 0.1, 0.2, 0.3 | Regularización. Por encima de 0.3 una red tan pequeña pierde capacidad |
| `lr` | log-uniforme [3·10⁻⁴, 3·10⁻³] | Una década alrededor de 10⁻³, el valor por defecto de Adam. Es el hiperparámetro más influyente |
| `weight_decay` | log-uniforme [10⁻⁶, 10⁻²] | Penalización L2 desacoplada (el alpha de Ridge): desde casi nada a fuerte |
| `batch_size` | 128, 256, 512 | Lotes pequeños dan más pasos por época y un ruido que regulariza; los grandes van más rápido en GPU |
| `dim_embedding` | 0 (target encoding), 8, 16 | Cómo entra el barrio (§2). Con ≈220 barrios, un embedding de 8–16 ≈ √220 |

**Fijos, no se buscan:** el optimizador (AdamW, que funciona bien sin ajuste fino, mientras que SGD necesita
ajustar momento y calendario), la pérdida (MSE, la del registro), las épocas (las decide la parada temprana,
máx. 300, paciencia 20) y la reducción de la tasa de aprendizaje al estancarse.

Los valores continuos se redondean a 3 cifras para que el nombre y la descripción del modelo en el registro
sean legibles.

In [ ]:
from scipy.stats import loguniform
from sklearn.model_selection import ParameterSampler

ESPACIO_MLP = {
    'capas': [(64,), (256,), (128, 64), (256, 128), (512, 256), (256, 128, 64), (512, 256, 128)],
    'activacion': ['relu', 'silu'],
    'dropout': [0.0, 0.1, 0.2, 0.3],
    'lr': loguniform(3e-4, 3e-3),
    'weight_decay': loguniform(1e-6, 1e-2),
    'batch_size': [128, 256, 512],
    'dim_embedding': [0, 8, 16],
}
N_ITER_MLP = 60  # fijado antes de ver resultados


def construir_mlp(nombre, params, **fijos):
    """Experimento con el preprocesado que corresponde a como entra el barrio (target o embedding)."""
    params = {**params, **fijos}
    barrio = 'embedding' if params.get('dim_embedding') else 'target'
    return Experimento(nombre, make_pipeline(preprocesador_mlp(VARS_MLP, barrio), RegresorMLP(**params)), VARS_MLP)


def redondear(params):
    return {k: float(f'{v:.3g}') if isinstance(v, float) else v for k, v in params.items()}


def resumen_filas(nombre, res):
    return {'experimento': nombre, 'rmse_log_val': res['rmse_log_val'].mean(),
            'rmse_log_val_sd': res['rmse_log_val'].std(), 'rmse_log_train': res['rmse_log_train'].mean(),
            't_ajuste_s': res['t_ajuste_s'].mean()}


def buscar_mlp(prefijo, espacio, n_iter):
    filas = []
    for i, params in enumerate(ParameterSampler(espacio, n_iter=n_iter, random_state=SEMILLA)):
        params, nombre = redondear(params), f'{prefijo}_{i:02d}'
        res = evaluar_registrado(construir_mlp(nombre, params), train, FOLDS_MODELOS, RUTA_REGISTRO)
        # 'params' guarda el diccionario tal cual: leido de las columnas, pandas convertiria los int en np.int64
        filas.append({**params, **resumen_filas(nombre, res), 'params': params})
        print(f"{nombre}: RMSE {filas[-1]['rmse_log_val']:.4f} | {filas[-1]['t_ajuste_s']:.0f} s por fold", flush=True)
    return pd.DataFrame(filas).sort_values('rmse_log_val')


busqueda_mlp = buscar_mlp('M8_mlp', ESPACIO_MLP, N_ITER_MLP)
display(busqueda_mlp.drop(columns='params').head(10).round(4))

### 3.1 Sensibilidad a cada hiperparámetro

Cada punto es un candidato. Si un hiperparámetro no muestra tendencia, no importa en este rango. Si el mejor
valor está en el borde de un rango continuo, el rango se quedó corto (anotarlo, no ampliar a la vista del
resultado sin registrarlo).

In [ ]:
fig, ejes = plt.subplots(2, 4, figsize=(15, 6.5), sharey=True)
for ax, param in zip(ejes.flat, ESPACIO_MLP):
    valores = busqueda_mlp[param]
    if param in ('lr', 'weight_decay'):
        ax.scatter(valores, busqueda_mlp['rmse_log_val'], s=18)
        ax.set_xscale('log')
    else:
        categorias = sorted(valores.astype(str).unique(), key=lambda c: (len(c), c))
        posicion = valores.astype(str).map({c: i for i, c in enumerate(categorias)})
        ax.scatter(posicion + np.random.default_rng(0).uniform(-0.15, 0.15, len(posicion)),
                   busqueda_mlp['rmse_log_val'], s=18)
        ax.set_xticks(range(len(categorias)), categorias, rotation=30, fontsize=8)
    ax.set_title(param)
ejes.flat[-1].axis('off')
ejes[0, 0].set_ylabel('RMSE log (media 5 folds)')
ejes[1, 0].set_ylabel('RMSE log (media 5 folds)')
fig.tight_layout()
plt.show()

## 4. RUIDO DE SEMILLA: LOS FINALISTAS CON TRES SEMILLAS

Una red da un resultado distinto con cada semilla (inicialización, orden de los lotes, apartado de parada).
Con 60 candidatos, el mejor por una sola semilla puede serlo en parte por suerte (*winner's curse*). Por eso los
**5 mejores** se reentrenan con otras dos semillas y se elige por la **media de las tres**. Las semillas extra
(142, 242) no se solapan con las del ensemble de §5 (42–46).

In [ ]:
N_FINALISTAS = 5
SEMILLAS_EXTRA = [SEMILLA + 100, SEMILLA + 200]

filas = []
for _, candidato in busqueda_mlp.head(N_FINALISTAS).iterrows():
    params = candidato['params']
    rmse = [candidato['rmse_log_val']]
    for semilla in SEMILLAS_EXTRA:
        res = evaluar_registrado(construir_mlp(f"{candidato['experimento']}_s{semilla}", params, semilla=semilla),
                                 train, FOLDS_MODELOS, RUTA_REGISTRO)
        rmse.append(res['rmse_log_val'].mean())
    filas.append({'experimento': candidato['experimento'], **params, 'rmse_semilla_42': rmse[0],
                  'rmse_media_3_semillas': np.mean(rmse), 'sd_entre_semillas': np.std(rmse, ddof=1),
                  'params': params})

finalistas = pd.DataFrame(filas).sort_values('rmse_media_3_semillas')
display(finalistas.drop(columns='params').round(4))

GANADOR = finalistas.iloc[0]
PARAMS_MLP = GANADOR['params']
print('Configuracion elegida:', GANADOR['experimento'], PARAMS_MLP)

## 5. MODELO FINAL Y COMPARACIÓN CON LOS DEMÁS

- `MLP_ajustado`: la configuración elegida, con una semilla.
- `MLP_ajustado_ens5`: la misma configuración entrenada con 5 semillas, prediciendo con la media. Promediar
  redes reduce la varianza de la predicción. Cuesta 5 veces más, así que solo compensa si la mejora es clara.

La comparación es **pareada por fold** con `comparar_pareado`, igual que en el resto del proyecto:
Δ = RMSE_otro − RMSE_MLP, así que un Δ positivo significa que el MLP gana. «Entra» exige mejora en ≥ 4 de 5 folds
y un Δ medio mayor que su desviación típica.

In [ ]:
mlp_final = evaluar_registrado(construir_mlp('MLP_ajustado', PARAMS_MLP), train, FOLDS_MODELOS, RUTA_REGISTRO)
mlp_ens = evaluar_registrado(construir_mlp('MLP_ajustado_ens5', PARAMS_MLP, n_semillas=5), train, FOLDS_MODELOS,
                             RUTA_REGISTRO)

REFERENCIAS = ['HGB_ajustado', 'RF_ajustado', 'S8_svr_gamma0.005_C1.0', 'T7_elasticnet_01']
referencia = pd.read_csv(RUTA_REFERENCIA).query('experimento in @REFERENCIAS')
referencia = referencia[referencia['repeticion'] == 0]
assert set(referencia['experimento']) == set(REFERENCIAS), 'Faltan modelos de referencia en el registro'

todos = pd.concat([referencia, mlp_final, mlp_ens], ignore_index=True)
display(resumir(todos).sort_values('rmse_log_val_mean').round(4))

comparaciones = {f'{mlp} frente a {otro}': comparar_pareado(res_mlp, referencia.query('experimento == @otro'))
                 for mlp, res_mlp in [('MLP_ajustado', mlp_final), ('MLP_ajustado_ens5', mlp_ens)]
                 for otro in REFERENCIAS}
display(pd.DataFrame(comparaciones).T)

## 6. CONCLUSIÓN

*Pendiente de completar con los resultados: la tabla de §5, si el MLP entra frente a HGB y frente al SVR,
cuánto aporta el ensemble, qué hiperparámetros importaron (§3.1) y el coste de ajuste frente a HGB.* La decisión
se anota en `docs/bitacora_decisiones.md`.